# Import data and packages

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
SRC_PATH = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_PATH))

import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from collections import defaultdict
from ntcp_fit.utilities import sigmoid

from SNPs_selection.snps_selection import run_bootstrap_single
from synth_data.outcome_generation import calibrate_intercept

In [41]:
DATA_DIR = PROJECT_ROOT / "data"

data_path = DATA_DIR / "synth_data_1000pts.xlsx"

data = pd.read_excel(data_path)

SNPs_list = ['SNP_1', 'SNP_2', 'SNP_3', 'SNP_4']

Generate a new outcome with an explicit and strong dependence on `SNP_1` and `SNP_2`, while keeping it independent of `SNP_3` and `SNP_4`. 

The final ranking is therefore expected to place `SNP_1` and `SNP_2` substantially above the other two variants, producing a clear elbow-shaped pattern that separates the relevant SNPs from the non-associated ones.

In [42]:
scores = - 0.8 * data['SNP_1'] + 0.7 * data['SNP_2']
beta_0 = calibrate_intercept(scores, 0.5)

probabilities = sigmoid(beta_0 + scores)

data['outcome_new'] = np.random.binomial(1, probabilities)

# Run bootstrap ranking

In [43]:
B = 1000

bootstrap_res = Parallel(n_jobs=-1, verbose=0)(
    delayed(run_bootstrap_single)(data, SNPs_list, "outcome_new", b) for b in range(B)
)

# Visualize the results

In [ ]:
bootstrap_dict = {'outcome_new': bootstrap_res}

for key in bootstrap_dict.keys():
    rank_dict = defaultdict(list)

    for boot in bootstrap_dict[key]:
        for snp, rank in boot:
            rank_dict[snp].append(rank)

    ranking_score = {snp: 1 - (np.mean(rank_dict[snp]) - 1)/(len(SNPs_list)-1) for snp in rank_dict.keys()}
    
    ordered_snps = sorted(ranking_score.items(), key=lambda x: -x[1])

    print(f"\nRanking {key}:")
    print(f"{'SNP':<20} {'Ranking Score':>10}")
    for snp, mr in ordered_snps:
        print(f"{snp:<20} {mr:>10.2f}")


Ranking outcome_new:
SNP                  Ranking Score
SNP_1                      0.95
SNP_2                      0.72
SNP_4                      0.23
SNP_3                      0.10


The final ranking is consistent with how the outcome was defined: `SNP_1` and `SNP_2` receive high ranking scores, whereas `SNP_3` and `SNP_4` have markedly lower scores, producing the expected elbow-shaped separation.
